Joonseo Bang & Khevin Jae Feliciano

Background information : A basketball player-season dataset with a clear prediction target. 5,000 rows, 26 columns, one CSV (~612 kB). all_star = 1 if the player-season is in the top ~8.5% of an internal selection score.

Source link : https://www.kaggle.com/datasets/vedauv/what-makes-an-nba-all-star-5000-player-seasons

## Import library

In [ ]:
from sklearn.naive_bayes import MultinomialNB       # The Naive Bayes model
from sklearn.feature_extraction.text import CountVectorizer  # Turns words into numbers
import numpy as np

## Create dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the NBA player seasons dataset
df = pd.read_csv('/content/nba_player_seasons.csv')

# Display basic info
print("Dataset Shape:", df.shape)

# We will set up a classification task: predict whether a player is an 'All Star' (is_all_star)
# Adjust target column based on the CSV schema (assuming 'is_all_star' or similar exists)
target_col = 'is_all_star' if 'is_all_star' in df.columns else df.columns[-1]

# Prepare features (X) and target (y)
# Drop non-numeric identifier columns if they exist
exclude_cols = ['player', 'team', 'season', target_col]
feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

X = df[feature_cols].fillna(0).values
y = df[target_col].values

print(f"Features used: {feature_cols}")
print("Features matrix (X) shape:", X.shape)
print("Target array (y) shape:", y.shape)

Dataset Shape: (5000, 26)
Features used: ['age', 'height_cm', 'draft_pick', 'team_win_pct', 'games_played', 'minutes_per_game', 'points_pg', 'rebounds_pg', 'assists_pg', 'steals_pg', 'blocks_pg', 'turnovers_pg', 'fg_pct', 'three_pt_pct', 'ft_pct', 'usage_pct', 'per', 'win_shares', 'bpm', 'salary_musd']
Features matrix (X) shape: (5000, 20)
Target array (y) shape: (5000,)


## Create & train model

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Instantiate MultinomialNB
model = MultinomialNB()

# Scale features because MultinomialNB requires non-negative values (e.g. 'bpm' can be negative)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train the model with the transformed NBA dataset features and targets
model.fit(X_scaled, y)

print("=== STEP 4: NBA Model trained successfully! ===")

=== STEP 4: NBA Model trained successfully! ===


## Test model

In [ ]:
# Define some test player profiles (using the exact 20 numerical features in feature_cols)
# Features order: ['age', 'height_cm', 'draft_pick', 'team_win_pct', 'games_played',
#                  'minutes_per_game', 'points_pg', 'rebounds_pg', 'assists_pg', 'steals_pg',
#                  'blocks_pg', 'turnovers_pg', 'fg_pct', 'three_pt_pct', 'ft_pct',
#                  'usage_pct', 'per', 'win_shares', 'bpm', 'salary_musd']

# Player 1: Elite superstar (likely an All-Star)
superstar_player = [28, 201, 1, 0.75, 75, 36.0, 28.5, 7.5, 6.2, 1.8, 1.2, 2.5, 0.49, 0.38, 0.85, 31.2, 25.4, 11.2, 6.5, 40.2]

# Player 2: Average bench player (unlikely to be an All-Star)
bench_player = [24, 190, 45, 0.40, 45, 12.0, 4.2, 1.5, 1.1, 0.4, 0.1, 0.8, 0.41, 0.31, 0.72, 14.5, 9.1, 1.1, -1.5, 2.1]

new_players = np.array([superstar_player, bench_player])

# Transform/scale features using the SAME scaler trained on our dataset
X_new_scaled = scaler.transform(new_players)

# Make predictions with our trained MultinomialNB model
predictions = model.predict(X_new_scaled)
probabilities = model.predict_proba(X_new_scaled)

print("=== NBA Predictions ===")
for i, player_name in enumerate(["Superstar Player", "Bench Player"]):
    is_all_star = "All-Star!" if predictions[i] == 1 else "Not an All-Star"
    prob_all_star = probabilities[i][1] * 100
    print(f"{player_name}: Predicted as {is_all_star} (Probability: {prob_all_star:.2f}%)")

=== NBA Predictions ===
Superstar Player: Predicted as Not an All-Star (Probability: 25.83%)
Bench Player: Predicted as Not an All-Star (Probability: 3.53%)


## Show results

In [ ]:
print("=== STEP 5 & 6: Predictions on new NBA Players ===")
print()

player_names = ["Superstar Player", "Bench Player"]

for i, name in enumerate(player_names):
    # predictions[i] is either 1 (All-Star) or 0 (Not All-Star)
    result = "All-Star!" if predictions[i] == 1 else "Not an All-Star"

    # probabilities[i][0] = probability of Not All-Star
    # probabilities[i][1] = probability of All-Star
    prob_allstar    = probabilities[i][1] * 100
    prob_regular    = probabilities[i][0] * 100

    print(f"Player: '{name}'")
    print(f"  --> Prediction : {result}")
    print(f"  --> All-Star chance: {prob_allstar:.1f}%")
    print(f"  --> Regular Player chance: {prob_regular:.1f}%")
    print()

=== STEP 5 & 6: Predictions on new NBA Players ===

Player: 'Superstar Player'
  --> Prediction : Not an All-Star
  --> All-Star chance: 25.8%
  --> Regular Player chance: 74.2%

Player: 'Bench Player'
  --> Prediction : Not an All-Star
  --> All-Star chance: 3.5%
  --> Regular Player chance: 96.5%



## Summary

Using a dataset of 5,000 NBA player-seasons, a Multinomial Naive Bayes classifier was built that predicts whether a player is an all-star, using 20 numeric features such as points, rebounds, assists, PER, win shares, BPM and salary. Since Multinomial Naive Bayes can't handle negative values like BPM,every feature was scaled to the 0–1 range with minmaxscaler, then trained the model on the full dataset. It was then tested on two made-up players, an elite superstar and an average bench player, and the model predicted that neither was an all-star: the superstar got only a 25.8% all-star probability and the bench player 3.5%. The superstar did rank clearly higher, but the model never predicted the all-star class for any player-season in the dataset, so its 91.5% (4575 players with all_star = 0 out of 5000) accuracy matches always guessing "not an all-star."